# XMCD Example Notebook

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import chan_vese

from imaging_toolbox.ptychography import remove_ramp_and_unwrap_phase
from imaging_toolbox.alignment import align_data, fft_shift_data
from imaging_toolbox.utils import normalise_data

In [104]:
def segmentation(
        data,
        y_range=(0, -1),
        x_range=(0, -1),
        circle=True,
        circle_y_range=(75, 100),
        circle_x_range=(25, 50)
        ):
    """ 
    Function to segment an image based on the chan vese algorithm.
    Can only be applied on arrays with real values
    """
    labels = chan_vese(
        data[y_range[0]:y_range[1],
             x_range[0]:x_range[1]],
             mu=0.05,
             init_level_set='checkerboard',
             dt=0.5
             )
    if circle==True:
        if np.average(
            labels[circle_y_range[0]:circle_y_range[1],
                   circle_x_range[0]:circle_x_range[1]]
            ) == 1:
            labels = 1-labels
    return labels

## Load LCP Data

In [ ]:
crop = 200

with h5py.File("lcp_ptycho_reconstruction.ptyr", "r") as f:
    lcp_data = np.array(f["content/obj/Sscan_00G00/data"][0,crop:-crop,crop:-crop])
    lcp_amp = np.abs(lcp_data)
    lcp_phase = np.angle(lcp_data)

## Choose Feature for Alignment

In [ ]:
align_x1 = 150
align_x2 = 575
align_y1 = 80
align_y2 = 550

fig, ax = plt.subplots(figsize=(8,8))

ax.imshow(lcp_amp, cmap="bone", vmin=0, vmax=2)
ax.set_title("Alignment ROI")
ax.axis("off")
ax.axvline(align_x1,color='r')
ax.axvline(align_x2,color='r')
ax.axhline(align_y1,color='b')
ax.axhline(align_y2,color='b')

plt.show()

# Choose Region for Phase Ramp Removal

In [ ]:
ramp_x1 = 180
ramp_x2 = 400
ramp_y1 = 120
ramp_y2 = 190

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16,8))

ax1.imshow(lcp_phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax1.set_title("Phase Ramp ROI")
ax1.axis("off")
ax1.axvline(ramp_x1,color='r')
ax1.axvline(ramp_x2,color='r')
ax1.axhline(ramp_y1,color='b')
ax1.axhline(ramp_y2,color='b')

rm_mask = np.zeros_like(lcp_amp, dtype="bool")
rm_mask[ramp_y1:ramp_y2, ramp_x1:ramp_x2] = True
ramp_corr_lcp_phase = remove_ramp_and_unwrap_phase(lcp_phase, mask=rm_mask)

ax2.imshow(ramp_corr_lcp_phase, cmap="bone", vmin=-3*np.pi, vmax=3*np.pi)
ax2.set_title("Corrected")
ax2.axis("off")

plt.show()

# Choose Area for Normalisation

In [ ]:
norm_x1 = 150
norm_x2 = 200
norm_y1 = 60
norm_y2 = 80

fig,(ax1,ax2,ax3) = plt.subplots(nrows=1, ncols=3, figsize=(24,8))

ax1.imshow(ramp_corr_lcp_phase.real, cmap='bone', vmin=-1.5*np.pi,vmax=1.5*np.pi)
ax1.axvline(norm_x1,color='r')
ax1.axvline(norm_x2,color='r')
ax1.axhline(norm_y1,color='b')
ax1.axhline(norm_y2,color='b')
ax1.set_title('Normalisation ROI')
ax1.axis("off")

norm_lcp_amp = normalise_data(
  lcp_amp,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="amplitude"
)

ax2.imshow(norm_lcp_amp, cmap="bone", vmin=0.2, vmax=1)
ax2.set_title("Normalised LCP Amp")
ax2.axis("off")

norm_lcp_phase = normalise_data(
  ramp_corr_lcp_phase.real,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="phase"
)

ax3.imshow(norm_lcp_phase, cmap="bone", vmin=-1, vmax=1)
ax3.set_title("Normalised LCP Phase")
ax3.axis("off")

plt.show()

# Load RCP Data

In [ ]:
with h5py.File("rcp_ptycho_reconstruction.ptyr", "r") as f:
    rcp_data = np.array(f["content/obj/Sscan_00G00/data"][0,crop:-crop,crop:-crop])
    rcp_amp = np.abs(rcp_data)
    rcp_phase = np.angle(rcp_data)

# Phase Ramp Removal

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16,8))

ax1.imshow(rcp_phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax1.set_title("Phase Ramp ROI")
ax1.axis("off")
ax1.axvline(ramp_x1,color='r')
ax1.axvline(ramp_x2,color='r')
ax1.axhline(ramp_y1,color='b')
ax1.axhline(ramp_y2,color='b')

rm_mask = np.zeros_like(rcp_amp, dtype="bool")
rm_mask[ramp_y1:ramp_y2, ramp_x1:ramp_x2] = True
ramp_corr_rcp_phase = remove_ramp_and_unwrap_phase(rcp_phase, mask=rm_mask)

ax2.imshow(ramp_corr_rcp_phase, cmap="bone", vmin=-3*np.pi, vmax=3*np.pi)
ax2.set_title("Corrected")
ax2.axis("off")

plt.show()

# Normalisation

In [ ]:
fig,(ax1,ax2,ax3) = plt.subplots(nrows=1, ncols=3, figsize=(24,8))

ax1.imshow(ramp_corr_rcp_phase.real, cmap='bone', vmin=-1.5*np.pi,vmax=1.5*np.pi)
ax1.axvline(norm_x1,color='r')
ax1.axvline(norm_x2,color='r')
ax1.axhline(norm_y1,color='b')
ax1.axhline(norm_y2,color='b')
ax1.set_title('Normalisation ROI')
ax1.axis("off")

norm_rcp_amp = normalise_data(
  rcp_amp,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="amplitude"
)

ax2.imshow(norm_rcp_amp, cmap="bone", vmin=0.2, vmax=1)
ax2.set_title("Normalised RCP Amp")
ax2.axis("off")

norm_rcp_phase = normalise_data(
  ramp_corr_rcp_phase.real,
  y_range=(norm_y1,norm_y2),
  x_range=(norm_x1,norm_x2),
  dtype="phase"
)

ax3.imshow(norm_rcp_phase, cmap="bone", vmin=-1, vmax=1)
ax3.set_title("Normalised RCP Phase")
ax3.axis("off")

plt.show()

# Alignment

In [112]:
seg_lcp_amp = segmentation(norm_lcp_amp[align_y1:align_y2, align_x1:align_x2])
seg_rcp_amp = segmentation(norm_rcp_amp[align_y1:align_y2, align_x1:align_x2])

_, shift = align_data(
  data=seg_rcp_amp,
  reference=seg_lcp_amp,
  normalisation="phase"
)

aligned_rcp_amp = fft_shift_data(data=norm_rcp_amp, shift=shift)
aligned_rcp_phase = fft_shift_data(data=norm_rcp_phase, shift=shift)

# XMCD Calculation

In [ ]:
crop=5

xmcd_amp = np.log(norm_lcp_amp) - np.log(aligned_rcp_amp)
xmcd_phase = norm_lcp_phase - aligned_rcp_phase

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(16,8))

ax1.imshow(xmcd_amp[crop:-crop, crop:-crop], cmap="bone", vmin=-0.2, vmax=0.2, alpha=1)
ax1.set_title("$A_{XMCD}$")
ax1.axis("off")

ax2.imshow(xmcd_phase[crop:-crop, crop:-crop], cmap="bone", vmin=-0.3, vmax=0.3)
ax2.set_title("$\\Phi_{XMCD}$")
ax2.axis("off")

plt.show()